# Physics Chatbot - Colab T4 Training Notebook

This notebook is now tuned for Google Colab T4 GPU training with QLoRA.

Steps:
1. Enable a GPU runtime in Colab
2. Clone the repo
3. Install requirements.txt
4. Verify CUDA
5. Login to Hugging Face
6. Download and prepare datasets
7. Train with QLoRA
8. Save the adapter to Drive
9. Run a quick smoke test


## 1. Clone the repo

In [ ]:
%cd /content
!rm -rf Sara-Phy-Chatbot
!git clone https://github.com/harkarshaurya-eng/Sara-Phy-Chatbot.git
%cd /content/Sara-Phy-Chatbot
!ls

## 2. Install packages

In [ ]:
!python -m pip install --upgrade pip -q
!pip install -r requirements.txt

## 3. Verify GPU

In [ ]:
!nvidia-smi

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected")

## 4. Login to Hugging Face

In [ ]:
from huggingface_hub import login
login()

## 5. Check environment

In [ ]:
!python scripts/00_check_env.py --config config.yaml

## 6. Download datasets

In [ ]:
!python scripts/01_download_datasets.py --config config.yaml

## 7. Prepare datasets

In [ ]:
!python scripts/02_prepare_dataset.py --config config.yaml

In [ ]:
!ls data/final

## 8. Train on T4 with QLoRA

In [ ]:
!python scripts/03_train_tpu.py --config config.yaml --num-train-epochs 30

## 9. Save outputs to Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import shutil

dst = '/content/drive/MyDrive/physics-chatbot-exports'
os.makedirs(dst, exist_ok=True)
shutil.copytree('outputs/adapters/final', f'{dst}/adapter_final', dirs_exist_ok=True)
for name in ['train_metrics.json', 'train_summary.md', 'data_report.md']:
    src = f'outputs/logs/{name}'
    if os.path.exists(src):
        shutil.copy2(src, f'{dst}/{name}')
print(f'Saved artifacts to {dst}')

## 10. Quick smoke test

In [ ]:
import sys
sys.path.insert(0, '.')

from src.inference import load_chat_model, generate_chat_response
from src.train_utils import load_config, get_system_prompt

config = load_config('config.yaml')
model, tokenizer, runtime = load_chat_model(
    base_model_name=config['base_model'],
    adapter_path='outputs/adapters/final',
    trust_remote_code=True,
    load_in_4bit=bool(config.get('load_in_4bit', False)),
)
result = generate_chat_response(
    model=model,
    tokenizer=tokenizer,
    model_name=config['base_model'],
    messages=[{'role': 'user', 'content': "Explain Newton's second law."}],
    temperature=0.7,
    max_new_tokens=256,
    system_prompt=get_system_prompt(config),
)
print(result['text'])